# 01 — Use correlation to ask better causal questions

Notebook 00 created a synthetic observational dataset with several different causal roles.

In this notebook, pretend you **do not know the generating DAG**.

Your task is to explore what the observed data can tell you, then create a worksheet of causal hypotheses for notebook 02.

### Tutorial path

00 Data preparation → **01 Correlational analysis** → 02 Causal inference

A key challenge in this example is intentional: a confounder, mediator, collider, instrument candidate, proxy, and irrelevant variable can all produce very different—and sometimes misleading—correlation patterns.


## 1. Configure the exploration

All plots use Seaborn.

- Spearman correlation summarizes monotonic marginal association.
- `vlag` is used for signed correlation plots so positive and negative values are easy to distinguish.
- Pairplots use a reproducible sample when the dataset is large.


In [ ]:
def default_params():
    return {
        "causal_dataset": "data/causal_data.csv",
        "dag_worksheet_output": "data/dag_worksheet.csv",
        "treatment_column": "treatment",
        "outcome_column": "output",
        "covariate_columns": [
            "code_number_tokens",
            "code_complexity",
            "code_num_identifiers",
            "code_num_strings",
            "developer_experience",
            "rollout_eligibility",
            "noise_feature",
            "docstring_detail_score",
            "review_flag",
        ],
        "correlation_method": "spearman",
        "plot_palette": "mako",
        "heatmap_palette": "vlag",
        "top_variables_to_plot": 7,
        "pairplot_max_rows": 3000,
        "random_seed": 42,
    }

params = default_params()
params


## 2. Load the table produced by notebook 00

At this stage we use only the observed data. The ground-truth DAG is intentionally not loaded.


In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from src.correlational_analysis_utils import (
    build_dag_worksheet,
    correlation_matrix,
    covariate_quality_table,
    load_causal_dataset,
    outcome_by_treatment_table,
    treatment_balance_table,
    variable_association_table,
)

sns.set_theme(style="whitegrid")

analysis_df, columns = load_causal_dataset(params)

treatment = columns.treatment
outcome = columns.outcome
covariates = columns.covariates

print(f"Rows: {len(analysis_df):,}")
print(f"Covariates: {len(covariates)}")

analysis_df.head()


## 3. Check whether every variable contains empirical variation

Correlation is undefined for a constant variable. This check prevents data-quality problems from being mistaken for substantive findings.


In [ ]:
quality_table = covariate_quality_table(
    analysis_df,
    covariates,
)

informative_covariates = (
    quality_table
    .loc[quality_table["has_variation"], "variable"]
    .tolist()
)

constant_covariates = (
    quality_table
    .loc[~quality_table["has_variation"], "variable"]
    .tolist()
)

print(f"Varying covariates: {len(informative_covariates)}")
print(f"Constant covariates: {constant_covariates or 'none'}")

quality_table


## 4. What is the raw treatment–outcome difference?

This is the association we would observe before adjusting for anything.

It motivates the causal question, but it is not automatically the treatment effect because treated and untreated units may differ for other reasons.


In [ ]:
outcome_summary = outcome_by_treatment_table(
    analysis_df,
    treatment=treatment,
    outcome=outcome,
)

control_mean = float(
    outcome_summary.loc[
        outcome_summary[treatment] == 0,
        "outcome_mean",
    ].iloc[0]
)
treated_mean = float(
    outcome_summary.loc[
        outcome_summary[treatment] == 1,
        "outcome_mean",
    ].iloc[0]
)

print(
    "Raw treated - control difference: "
    f"{treated_mean - control_mean:.3f}"
)

outcome_summary


In [ ]:
plt.figure(figsize=(6, 4))

sns.barplot(
    data=analysis_df,
    x=treatment,
    y=outcome,
    hue=treatment,
    palette=params["plot_palette"],
    errorbar=("ci", 95),
    legend=False,
)

plt.title("Observed outcome by treatment group")
plt.xlabel("Treatment")
plt.ylabel("Mean outcome")
plt.tight_layout()
plt.show()


## 5. Which variables move together?

The heatmap is useful for finding clusters, proxies, and variables associated with treatment or outcome.

A large value tells you **that two variables move together**, not which one causes the other.


In [ ]:
analysis_columns = [
    treatment,
    outcome,
    *informative_covariates,
]

corr = correlation_matrix(
    analysis_df,
    columns=analysis_columns,
    method=params["correlation_method"],
)

corr.round(2)


In [ ]:
mask = np.triu(
    np.ones_like(corr, dtype=bool),
    k=1,
)

plt.figure(figsize=(12, 9))

sns.heatmap(
    corr,
    mask=mask,
    cmap=params["heatmap_palette"],
    center=0,
    vmin=-1,
    vmax=1,
    annot=True,
    fmt=".2f",
    square=True,
    linewidths=0.5,
    cbar_kws={
        "label": f"{params['correlation_method'].title()} correlation"
    },
)

plt.title("Pairwise associations among observed variables")
plt.tight_layout()
plt.show()


## 6. Which variables differ between treated and control units?

The standardized mean difference (SMD) describes treatment-group imbalance.

A variable can be strongly imbalanced because it:

- helps cause treatment;
- is caused by treatment;
- is correlated with a cause of treatment; or
- participates in another structure.

So imbalance is a clue, not a causal label.


In [ ]:
balance_table = treatment_balance_table(
    analysis_df,
    treatment=treatment,
    covariates=covariates,
)

balance_table.round(3)


In [ ]:
balance_plot = (
    balance_table
    .loc[balance_table["has_variation"]]
    .sort_values("standardized_mean_difference")
)

plt.figure(figsize=(9, 6))

sns.barplot(
    data=balance_plot,
    x="standardized_mean_difference",
    y="variable",
    hue="variable",
    palette=params["plot_palette"],
    legend=False,
)

plt.axvline(0, color="black", linewidth=1)
plt.axvline(-0.10, color="gray", linestyle="--", linewidth=1)
plt.axvline(0.10, color="gray", linestyle="--", linewidth=1)

plt.title("Covariate imbalance between treatment groups")
plt.xlabel("Standardized mean difference")
plt.ylabel("")
plt.tight_layout()
plt.show()


## 7. Map each variable against treatment and outcome

This view is especially useful for DAG discussion.

Variables near the upper-right or lower-left corners are strongly associated with both treatment and outcome. But that pattern can arise from very different causal roles—including a confounder, mediator, or collider.

A variable strongly related to treatment but weakly related to outcome could be an instrument candidate, treatment predictor, proxy, or simply weakly connected to outcome.

Again, the plot raises questions; it does not answer them.


In [ ]:
association_table = variable_association_table(
    analysis_df,
    treatment=treatment,
    outcome=outcome,
    covariates=informative_covariates,
    method=params["correlation_method"],
)

association_table.round(3)


In [ ]:
plt.figure(figsize=(9, 7))

ax = sns.scatterplot(
    data=association_table,
    x="association_with_treatment",
    y="association_with_outcome",
    size="screening_score",
    hue="screening_score",
    palette=params["plot_palette"],
    sizes=(70, 320),
    legend=False,
)

plt.axvline(0, color="gray", linewidth=1)
plt.axhline(0, color="gray", linewidth=1)

for row in association_table.itertuples():
    ax.text(
        row.association_with_treatment + 0.008,
        row.association_with_outcome + 0.008,
        row.variable,
        fontsize=9,
    )

plt.title("Observed association with treatment and outcome")
plt.xlabel(f"Association with treatment ({params['correlation_method']})")
plt.ylabel(f"Association with outcome ({params['correlation_method']})")
plt.tight_layout()
plt.show()


### Pause before assigning roles

For each interesting point in the association map, ask:

- Was this variable determined before treatment?
- Could treatment cause it?
- Could outcome cause it?
- Could both treatment and outcome cause it?
- Could it affect treatment without directly affecting outcome?
- Could it simply be correlated with another measured cause?

Those questions are what separate DAG construction from variable screening.


## 8. Inspect a few strong relationships in more detail

A pairplot can reveal nonlinear structure and treatment-group overlap that one correlation coefficient hides.

For readability, the notebook samples at most `pairplot_max_rows` observations.


In [ ]:
top_n = min(
    int(params["top_variables_to_plot"]),
    len(association_table),
)

top_variables = (
    association_table
    .head(top_n)["variable"]
    .tolist()
)

pairplot_variables = top_variables[: min(4, len(top_variables))]
pairplot_n = min(
    int(params["pairplot_max_rows"]),
    len(analysis_df),
)

if pairplot_variables:
    pairplot_df = (
        analysis_df[
            [treatment, *pairplot_variables]
        ]
        .sample(
            n=pairplot_n,
            random_state=params["random_seed"],
        )
    )

    sns.pairplot(
        pairplot_df,
        vars=pairplot_variables,
        hue=treatment,
        palette=sns.color_palette(
            params["plot_palette"],
            n_colors=2,
        ),
        corner=True,
        diag_kind="hist",
        plot_kws={
            "alpha": 0.50,
            "s": 22,
        },
    )
    plt.show()
else:
    print("No varying covariates available for the pairplot.")


## 9. Build the DAG worksheet

The worksheet is the handoff to notebook 02.

It copies the observed evidence into one table and leaves the causal decisions blank:

- `proposed_role`
- `proposed_edges`
- `domain_justification`

Do not fill those columns by ranking correlations. Use temporal order and a plausible mechanism.


In [ ]:
dag_worksheet = build_dag_worksheet(
    association_table=association_table,
    balance_table=balance_table,
    quality_table=quality_table,
    treatment=treatment,
    outcome=outcome,
)

dag_worksheet


## 10. Questions to answer before notebook 02

For every variable, try to decide:

1. Was it determined before treatment?
2. Can it cause treatment?
3. Can it cause outcome?
4. Can treatment cause it?
5. Can outcome cause it?
6. Is it mostly a proxy for another variable?
7. Would conditioning on it block a causal pathway we care about?
8. Could conditioning on it open a collider path?
9. Is there a credible mechanism for each proposed arrow?

You do not need to be certain. The point is to make the assumptions explicit.


In [ ]:
from pathlib import Path

worksheet_path = Path(params["dag_worksheet_output"])
worksheet_path.parent.mkdir(parents=True, exist_ok=True)
dag_worksheet.to_csv(worksheet_path, index=False)

print(f"Saved DAG worksheet to: {worksheet_path}")


## Next: 02 — Build and test the causal model

Bring the worksheet into notebook 02 and construct the DAG **before revealing the synthetic ground truth**.

After you make your graph, notebook 02 will let you compare it with the data-generating DAG. That comparison is the central lesson of the example:

> Variables with similar correlations can have very different causal roles.
